In [6]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../Master-Data', exist_ok=True)

# Load Sprint 2 output
df = pd.read_csv('../Master-Data/categorized_food_data.csv', low_memory=False)


ns_col = 'nutriscore_score'
if ns_col not in df.columns:
    clean_df = pd.read_csv('../Master-Data/clean_food_data.csv',
                           usecols=['product_name', 'sugars_100g',
                                    'proteins_100g', ns_col],
                           low_memory=False)
    df = df.merge(clean_df, on=['product_name', 'sugars_100g', 'proteins_100g'], how='left')
    print(f'Nutri-score merged: {df[ns_col].notna().sum():,} products')

# Define thresholds consistent with Sprint 4
PROTEIN_THRESHOLD = 15
SUGAR_THRESHOLD   = 10
FIBER_THRESHOLD   = 5

print(f'Data loaded: {len(df):,} rows | {df["Primary_Category"].nunique()} categories')
df.head(3)

Data loaded: 47,495 rows | 16 categories


,product_name,categories_tags,ingredients_text,nutriscore_score,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,Primary_Category
0,Pinto Bean,en:asian-style-ready-meal,NaN,NaN,9.0,10.2,3.6,22.5,4.9,3.8,17.5,NaN,Other Snacks
1,pasta,"en:beverages-and-beverages-preparations,en:bev...",NaN,5.0,697.1,6.4,1.3,20.0,1.9,0.8,6.7,0.0009,Other Snacks
2,Eirn original curry Sauce,"en:plant-based-foods-and-beverages,en:plant-ba...",Wheat Flour Sugar Vegetable Fat D Curry Powder...,NaN,1724.0,18.0,8.2,57.2,29.6,1.2,5.2,4.7500,Other Snacks


---
## Candidate's Choice: Fiber Opportunity Analysis

### What I Added
A **Triple Opportunity Analysis** — identifying which snack categories are under-served
across three dimensions simultaneously: **High Protein + High Fiber + Low Sugar**.

### Why I Added It
The project brief explicitly states the client wants products that meet **High Protein AND
High Fiber** consumer demand. However Stories 3 and 4 only analyse Sugar vs Protein.
By adding fiber we:

1. **Directly address the brief** — fiber is mentioned in the business context but no one else is measuring it
2. **Stronger product claim** — 'High Protein + High Fiber + Low Sugar' commands a premium price
3. **Regulatory advantage** — meets EU criteria for both high protein AND high fiber front-of-pack claims
4. **Harder to copy** — a triple-claim product is much rarer and more defensible than a single-claim

In [7]:
# Flag each product zone
df['In_Healthy_Zone'] = (
    (df['proteins_100g'] >= PROTEIN_THRESHOLD) &
    (df['sugars_100g']   <= SUGAR_THRESHOLD)
)

df['In_Triple_Zone'] = (
    (df['proteins_100g']            >= PROTEIN_THRESHOLD) &
    (df['sugars_100g']              <= SUGAR_THRESHOLD)   &
    (df['fiber_100g'].fillna(0)     >= FIBER_THRESHOLD)
)

# Triple Opportunity per category
triple_results = []

for cat in sorted(df['Primary_Category'].unique()):
    cat_df         = df[df['Primary_Category'] == cat]
    total          = len(cat_df)
    in_healthy     = cat_df['In_Healthy_Zone'].sum()
    in_triple      = cat_df['In_Triple_Zone'].sum()
    pct_healthy    = round(in_healthy / total * 100, 1)
    pct_triple     = round(in_triple  / total * 100, 1)
    med_fiber      = round(cat_df['fiber_100g'].median(), 1)
    med_protein    = round(cat_df['proteins_100g'].median(), 1)
    med_sugar      = round(cat_df['sugars_100g'].median(), 1)

    triple_results.append({
        'Category'                    : cat,
        'Total Products'              : total,
        'In Healthy Zone'             : int(in_healthy),
        '% Healthy Zone'              : pct_healthy,
        'In Triple Zone'              : int(in_triple),
        '% Triple Zone'               : pct_triple,
        'Median Fiber (g)'            : med_fiber,
        'Median Protein (g)'          : med_protein,
        'Median Sugar (g)'            : med_sugar,
    })

triple_df = pd.DataFrame(triple_results).sort_values('% Triple Zone', ascending=True)

print('=== TRIPLE OPPORTUNITY INDEX ===')
print(f'Criteria: Protein >= {PROTEIN_THRESHOLD}g | Sugar <= {SUGAR_THRESHOLD}g | Fiber >= {FIBER_THRESHOLD}g')
print(f'Low % = category has very few products meeting all 3 = biggest gap')
print()
print(f'{"Category":<35} {"Total":>7}  {"% Healthy":>10}  {"% Triple":>10}  {"Med Fiber":>10}')
print('-' * 80)
for _, row in triple_df.iterrows():
    flag = '  ← BIGGEST TRIPLE GAP' if row['% Triple Zone'] == triple_df['% Triple Zone'].min() else ''
    print(f"{row['Category']:<35} {int(row['Total Products']):>7,}  {row['% Healthy Zone']:>9.1f}%  {row['% Triple Zone']:>9.1f}%  {row['Median Fiber (g)']:>9.1f}g{flag}")

print(f'\n Candidate\'s Choice analysis done')

=== TRIPLE OPPORTUNITY INDEX ===
Criteria: Protein >= 15g | Sugar <= 10g | Fiber >= 5g
Low % = category has very few products meeting all 3 = biggest gap

Category                              Total   % Healthy    % Triple   Med Fiber
--------------------------------------------------------------------------------
Fruit Snacks                             14        0.0%        0.0%        5.0g  ← BIGGEST TRIPLE GAP
Sweet Spreads & Dips                    655        2.6%        0.0%        0.0g  ← BIGGEST TRIPLE GAP
Meat & Fish Snacks                      158       51.9%        0.0%        0.0g  ← BIGGEST TRIPLE GAP
Yogurt & Dairy Snacks                 3,924        1.5%        0.0%        0.0g  ← BIGGEST TRIPLE GAP
Popcorn                                 123        0.0%        0.0%        8.5g  ← BIGGEST TRIPLE GAP
Ice Cream & Frozen Snacks               671        0.0%        0.0%        0.6g  ← BIGGEST TRIPLE GAP
Biscuits & Cookies                    1,776        1.1%        0.6%     

In [8]:
#  Key insight from Triple Analysis
best_triple_cat = triple_df.iloc[0]['Category']
best_triple_pct = triple_df.iloc[0]['% Triple Zone']
best_healthy_pct = triple_df.iloc[0]['% Healthy Zone']

print('╔══════════════════════════════════════════════════════════════════╗')
print('║         CANDIDATE\'S CHOICE — KEY FINDING                       ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print(f'║')
print(f'║  Category with biggest triple gap: {best_triple_cat}')
print(f'║')
print(f'║  Only {best_triple_pct:.1f}% of products in this category meet')
print(f'║  ALL THREE criteria: High Protein + High Fiber + Low Sugar')
print(f'║')
print(f'║  Even the 2-dimension gap is {best_healthy_pct:.1f}% — so adding fiber')
print(f'║  makes the opportunity even more exclusive.')
print(f'║')
print(f'║  A product with >= {PROTEIN_THRESHOLD}g protein, >= {FIBER_THRESHOLD}g fiber,')
print(f'║  and <= {SUGAR_THRESHOLD}g sugar would be virtually alone in this market.')
print('║')
print('╚══════════════════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════════════════╗
║         CANDIDATE'S CHOICE — KEY FINDING                       ║
╠══════════════════════════════════════════════════════════════════╣
║
║  Category with biggest triple gap: Fruit Snacks
║
║  Only 0.0% of products in this category meet
║  ALL THREE criteria: High Protein + High Fiber + Low Sugar
║
║  Even the 2-dimension gap is 0.0% — so adding fiber
║  makes the opportunity even more exclusive.
║
║  A product with >= 15g protein, >= 5g fiber,
║  and <= 10g sugar would be virtually alone in this market.
║
╚══════════════════════════════════════════════════════════════════╝


In [10]:
triple_df.to_csv('../Master-Data/powerbi_triple_opportunity.csv', index=False)
print('   Exported: ../Master-Data/powerbi_triple_opportunity.csv')
print(f'   Rows    : {len(triple_df)}')

   Exported: ../Master-Data/powerbi_triple_opportunity.csv
   Rows    : 16
